# Part 2 - Sentetik Soru Üretimi ve Veri Çoğaltma (Anthropic Claude API)

Bu notebook, bir önceki aşamada elde edilen 100 adet temel matematik problemini (GSM8K) kullanarak daha geniş ve çeşitli bir veri seti oluşturmayı amaçlamaktadır. Her bir orijinal soru için 5 farklı strateji kullanılarak 25'er adet yeni soru üretilmiş ve toplamda 2500 soruluk genişletilmiş (augmented) bir veri seti elde edilmiştir.

**Neden Gemini yerine Claude Kullanıldı?**
* **Kota ve Limitler:** Gemini Free Tier günde yalnızca kısıtlı istek hakkı tanımaktadır. 100 soru x 1 istek = 100 istek gerektiğinden kota hızla dolmaktadır.
* **Maliyet Etkinliği:** `claude-haiku-4-5` veya `claude-3-5-haiku-20241022` modellerinin maliyeti oldukça düşüktür ($0.80/MTok input, $4/MTok output). 100 soruluk bu operasyonun tahmini toplam maliyeti sadece **$0.10 - $0.30** civarındadır.
* **Hız ve Kapasite:** Rate limit çok daha yüksektir (Tier 1 için dakikada 50 istek), bu da toplu veri üretimini hızlandırır.

*(API Anahtarı almak için: https://console.anthropic.com/)*

## 1. Kurulum ve Bağlantılar
Gerekli kütüphaneleri yüklüyor ve veri setlerini kaydedebilmek için Google Drive bağlantımızı sağlıyoruz.

In [ ]:
# Bağımlılıkları yükle
!pip install -q anthropic pandas

# Google Drive bağlantısı (Google Colab ortamı için)
from google.colab import drive
drive.mount('/content/drive')

## 2. Kütüphanelerin İçe Aktarılması ve API Ayarları
Model ile iletişim kurmak için Anthropic istemcisini başlatıyoruz. Güvenlik amacıyla API anahtarının Colab Secrets üzerinden (veya ortam değişkenlerinden) alınması önerilir.

In [ ]:
import anthropic
import pandas as pd
import json
import time
import re
import os

# ============================================================
# API ANAHTARI AYARI
# ============================================================
try:
    from google.colab import userdata
    ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
    print("API anahtarı Colab Secrets'tan başarıyla alındı.")
except Exception:
    # Alternatif: Doğrudan giriş (Kodunuzu paylaşırken burayı gizlemeyi unutmayın!)
    ANTHROPIC_API_KEY = "sk-ant-api03-..."  # <-- API anahtarınızı buraya yapıştırın
    print("API anahtarı manuel olarak ayarlandı.")

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
print("Anthropic client hazır.")

# Model seçimi: En hızlı ve maliyet etkin model tercih edilmiştir.
MODEL = "claude-haiku-4-5"
print(f"Kullanılacak model: {MODEL}")

## 3. Orijinal Verilerin Yüklenmesi ve Checkpoint Mekanizması
Bir önceki notebook'ta başarıyla çözülen ve çözülemeyen toplam 100 soruyu yüklüyoruz. Ayrıca, uzun süren API işlemlerinde bağlantı kopması ihtimaline karşı bir "kaldığı yerden devam etme" (checkpoint) mekanizması kuruyoruz.

In [ ]:
# Önceki aşamadaki verileri yükle
print("100 orijinal soru yükleniyor...")
df_success = pd.read_csv("/content/drive/MyDrive/kaynaklar/qwen3_5_4b_basarili_50.csv")
df_fail = pd.read_csv("/content/drive/MyDrive/kaynaklar/qwen3_5_4b_basarisiz_50.csv")

# Verileri birleştir ve orijinal durumlarını etiketle
df_success['orijinal_durum'] = 'basarili'
df_fail['orijinal_durum'] = 'basarisiz'
df_all = pd.concat([df_success, df_fail], ignore_index=True)

print(f"Toplam {len(df_all)} soru yüklendi.\n")

# Checkpoint Kontrolü
BACKUP_FILE = "augmented_dataset_backup.csv"

try:
    existing_df = pd.read_csv(BACKUP_FILE)
    # Her sorudan 25 yeni soru ürettiğimiz için işlenen soru sayısını hesaplıyoruz
    islenen_soru_sayisi = len(existing_df) // 25
    augmented_data = existing_df.to_dict('records')
    print(f"Kaldığı yer bulundu. İlk {islenen_soru_sayisi} soru atlanıyor...")
    print(f"Şu ana kadar {len(augmented_data)} soru üretilmiş.")
except FileNotFoundError:
    islenen_soru_sayisi = 0
    augmented_data = []
    print("Yedek dosya bulunamadı. İşleme baştan başlanıyor...")

## 4. LLM Soru Üretim Fonksiyonu (Prompt Engineering & Error Handling)
Modelin her bir soru için 5 farklı strateji kullanarak toplam 25 soru üretmesini sağlayan ana fonksiyondur.

**Stratejiler:**
1. Tema Değişikliği
2. Karmaşıklaştırma
3. Sadeleştirme
4. Hikayeleştirme
5. Tersine Mühendislik

*Not: Fonksiyon içerisinde JSON parse hatalarına ve Rate Limit (429) sınırlarına karşı bekleme (exponential backoff) mekanizmaları bulunmaktadır.*

In [ ]:
def generate_questions_with_claude(original_question, original_status, max_retries=5):
    """
    Verilen orijinal sorudan 5 strateji x 5 soru = 25 soru üretir.
    Anthropic API kullanır, hata durumunda retry + exponential backoff uygular.
    """
    prompt = f"""Aşağıda sana verilen orijinal matematik problemini incele.
Orijinal Soru: "{original_question}"

Bu problemin matematiksel mantığını baz alarak aşağıda belirtilen 5 farklı stratejinin HER BİRİ için 5'er adet (Toplam 25 adet) yeni soru üret.

Stratejiler:
1. "Tema Değişikliği": İşlemleri birebir koru, nesneleri ve mekanı tamamen farklı bir bağlama taşı.
2. "Karmaşıklaştırma": Hikayeyi koru ancak sayıları daha büyük, ondalıklı veya karmaşık hale getir.
3. "Sadeleştirme": Hikaye ve gereksiz betimlemeleri at. Sadece saf işlem komutları içeren kısa bir soru bırak.
4. "Hikayeleştirme": Soruyu edebi bir metne dönüştür. Dikkat dağıtıcı ama işlemsiz detaylar ekle.
5. "Tersine Mühendislik": Orijinal sorunun sonucunu metne dahil et ve hikayedeki başlangıç değerini sor.

KURALLAR:
1. Ürettiğin sorular kesinlikle mantıklı ve çözülebilir olmalıdır.
2. Çıktını SADECE aşağıdaki geçerli JSON formatında ver, araya markdown veya metin ekleme!
3. Her strateji için tam olarak 5 soru olmalı (toplam 25 soru).

[
    {{"strateji": "Tema Değişikliği", "soru": "...", "beklenen_cevap": deger}},
    ...
]"""

    for attempt in range(max_retries):
        try:
            message = client.messages.create(
                model=MODEL,
                max_tokens=4096,
                temperature=0.7,
                messages=[{"role": "user", "content": prompt}]
            )

            response_text = message.content[0].text.strip()

            # JSON temizleme: Model cevabı markdown bloğu içine almış olabilir
            if response_text.startswith("```"):
                response_text = re.sub(r'^```[a-z]*\n?', '', response_text)
                response_text = re.sub(r'```$', '', response_text).strip()

            generated_list = json.loads(response_text)

            result = []
            for item in generated_list:
                result.append({
                    "original_question": original_question,
                    "original_status": original_status,
                    "strategy": item.get("strateji", "Bilinmeyen"),
                    "generated_question": item.get("soru", ""),
                    "expected_number": item.get("beklenen_cevap", 0)
                })
            return result

        except anthropic.RateLimitError as e:
            wait_time = 60 * (attempt + 1)
            print(f"  Rate limit aşıldı! {wait_time}s bekleniyor... (Deneme {attempt+1}/{max_retries})")
            time.sleep(wait_time)

        except anthropic.APIStatusError as e:
            wait_time = 10 * (attempt + 1)
            print(f"  API Hatası ({e.status_code}): {e.message}. {wait_time}s bekleniyor...")
            time.sleep(wait_time)

        except json.JSONDecodeError as e:
            print(f"  JSON parse hatası (Deneme {attempt+1}): {e}")
            time.sleep(5)

        except Exception as e:
            print(f"  Beklenmeyen hata: {e}. 10s bekleniyor...")
            time.sleep(10)

    print(f"  HATA: {max_retries} denemeden sonra başarısız. Bu soru atlanıyor.")
    return []

## 5. Ana Üretim Döngüsü
Tüm soruları sırayla işler. API sınırlarına takılmamak için her işlem arasına kısa bir bekleme süresi koyar ve her 5 soruda bir veriyi yedekler.

In [ ]:
toplam_soru = len(df_all)
print(f"\nVeri çoğaltma işlemi başlıyor. Toplam {toplam_soru} soru işlenecek.")
print(f"Tahmini süre: ~{toplam_soru * 3 // 60} dakika")
print(f"Tahmini maliyet (Haiku): ~${toplam_soru * 0.003:.2f}\n")

for index, row in df_all.iterrows():
    if index < islenen_soru_sayisi:
        continue  # Checkpoint'e göre işlenmişleri atla

    original_question = row['question']
    original_status = row['orijinal_durum']

    print(f"--- Soru {index + 1}/{toplam_soru} İşleniyor ---")
    new_questions = generate_questions_with_claude(original_question, original_status)

    if new_questions:
        augmented_data.extend(new_questions)
        print(f"  ✓ {len(new_questions)} soru üretildi. Toplam: {len(augmented_data)}")
    else:
        print(f"  ✗ Bu soru için üretim başarısız oldu.")

    # Rate Limit önlemi (Haiku Tier 1 limiti: 50 req/min)
    time.sleep(2)

    # Her 5 soruda bir yedek al (Backup)
    if (index + 1) % 5 == 0:
        pd.DataFrame(augmented_data).to_csv(BACKUP_FILE, index=False)
        print(f"** Yedek alındı: {len(augmented_data)} soru kaydedildi. **")

print("\nİlk aşama üretim tamamlandı!")

## 6. Eksik Verilerin Tespiti ve Tamamlanması
API hataları veya JSON parse problemleri nedeniyle bazı stratejiler eksik üretilmiş olabilir. Bu adımda veri seti taranır ve eksik olan stratejiler için hedefe yönelik spesifik bir prompt ile eksikler tamamlanır.

In [ ]:
# Elde edilen veriyi dataframe'e alalım
df_aug = pd.DataFrame(augmented_data)

stratejiler = [
    "Tema Değişikliği", "Karmaşıklaştırma", "Sadeleştirme",
    "Hikayeleştirme", "Tersine Mühendislik"
]

eksikler = []

for idx, row in df_all.iterrows():
    soru = row['question']
    durum = row['orijinal_durum']
    alt_df = df_aug[df_aug['original_question'] == soru]

    # Hiç üretilememişse
    if len(alt_df) == 0:
        eksikler.append({
            "soru_index": idx + 1, "original_status": durum,
            "strateji": "TÜMÜ", "mevcut": 0, "eksik": 25
        })
        continue

    # Strateji bazında eksik kontrolü
    for strateji in stratejiler:
        mevcut = len(alt_df[alt_df['strategy'] == strateji])
        if mevcut < 5:
            eksikler.append({
                "soru_index": idx + 1, "original_status": durum,
                "strateji": strateji, "mevcut": mevcut, "eksik": 5 - mevcut
            })

df_eksik = pd.DataFrame(eksikler)

if df_eksik.empty:
    print("✓ Tüm sorular eksiksiz üretilmiş!")
else:
    print(f"Tespit edilen eksiklik sayısı: {len(df_eksik)} (Toplam {df_eksik['eksik'].sum()} soru eksik)")

Spesifik eksikleri tamamlayan yardımcı fonksiyonlar:

strateji_aciklamalari = {
    "Tema Değişikliği": "İşlemleri birebir koru, nesneleri ve mekanı tamamen farklı bir bağlama taşı.",
    "Karmaşıklaştırma": "Hikayeyi koru ancak sayıları daha büyük, ondalıklı veya karmaşık hale getir.",
    "Sadeleştirme": "Hikaye ve gereksiz betimlemeleri at. Sadece saf işlem komutları içeren kısa bir soru bırak.",
    "Hikayeleştirme": "Soruyu edebi bir metne dönüştür. Dikkat dağıtıcı ama işlemsiz detaylar ekle.",
    "Tersine Mühendislik": "Orijinal sorunun sonucunu metne dahil et ve hikayedeki başlangıç değerini sor."
}

def parse_response(response_text):
    """JSON parse eder, markdown bloğu varsa temizler."""
    if response_text.startswith("```"):
        response_text = re.sub(r'^```[a-z]*\n?', '', response_text)
        response_text = re.sub(r'```$', '', response_text).strip()
    return json.loads(response_text)

def generate_specific(original_question, original_status, strateji, eksik_adet):
    """Sadece eksik olan belirli bir strateji için soru üretir."""
    prompt = f"""Aşağıdaki matematik problemini incele.
Orijinal Soru: "{original_question}"

Yalnızca "{strateji}" stratejisini kullanarak {eksik_adet} adet yeni soru üret.
Strateji: {strateji_aciklamalari[strateji]}

Çıktını SADECE geçerli JSON formatında ver:
[
    {{"strateji": "{strateji}", "soru": "...", "beklenen_cevap": 0}}
]
(toplam {eksik_adet} eleman olmalı)"""

    for attempt in range(5):
        try:
            message = client.messages.create(
                model=MODEL,
                max_tokens=8096,
                temperature=0.7,
                messages=[{"role": "user", "content": prompt}]
            )
            return parse_response(message.content[0].text.strip())
        except Exception as e:
            print(f"  Hata (Deneme {attempt+1}): {e}")
            time.sleep(5)
    return []

# Eksikleri tamamlama döngüsü
if not df_eksik.empty:
    print(f"Toplam {df_eksik['eksik'].sum()} eksik soru tamamlanıyor...\n")
    for _, eksik_row in df_eksik.iterrows():
        soru_idx = eksik_row['soru_index'] - 1
        strateji = eksik_row['strateji']
        eksik_adet = int(eksik_row['eksik'])
        original_question = df_all.iloc[soru_idx]['question']
        original_status = df_all.iloc[soru_idx]['orijinal_durum']

        if strateji == "TÜMÜ":
            print(f"Soru {eksik_row['soru_index']}: TÜMÜ üretiliyor...")
            yeni_sorular = generate_questions_with_claude(original_question, original_status)
        else:
            print(f"Soru {eksik_row['soru_index']} - {strateji}: {eksik_adet} soru üretiliyor...")
            result = generate_specific(original_question, original_status, strateji, eksik_adet)
            yeni_sorular = [{
                "original_question": original_question,
                "original_status": original_status,
                "strategy": item.get("strateji", strateji),
                "generated_question": item.get("soru", ""),
                "expected_number": item.get("beklenen_cevap", 0)
            } for item in result]

        if yeni_sorular:
            augmented_data.extend(yeni_sorular)
            print(f"  ✓ {len(yeni_sorular)} soru eklendi.")
        time.sleep(2)

## 7. Veri Temizliği ve Final Kaydı
Tüm sorular üretildikten sonra veri setindeki olası tipografik anormallikler temizlenir (Örn: Modelin strateji ismini yanlış yazması) ve son hali dışa aktarılır.

In [ ]:
DRIVE_PATH = "/content/drive/MyDrive/kaynaklar/augmented_dataset_final_2500.csv"
df_final = pd.DataFrame(augmented_data)

# Veri Temizliği: Örnek bir tipografi düzeltmesi (Eğer model stratejiyi yanlış yazdıysa silinir veya düzeltilir)
# Burada 2500 soru limitini korumak adına fazla üretilmiş satırları temizliyoruz.
hatali_mask = df_final['strategy'] == 'Karmaßıklaştırma'
if hatali_mask.any():
    df_final = df_final[~hatali_mask]

# Final csv kaydı
df_final.to_csv("augmented_dataset_final_2500.csv", index=False)
df_final.to_csv(DRIVE_PATH, index=False)

print(f"\nİşlem tamamlandı! Toplam {len(df_final)} soru üretildi ve Drive'a kaydedildi.")
print("\nStrateji dağılımı:")
print(df_final['strategy'].value_counts())
print("\nOrijinal durum dağılımı:")
print(df_final['original_status'].value_counts())

## Alternatif Yöntem: OpenRouter Kullanımı (İsteğe Bağlı)

Eğer Anthropic hesabı açmak istemiyorsanız, **OpenRouter** üzerinden tamamen ücretsiz modellere (Mistral, Llama, Gemma) erişebilirsiniz. Aşağıdaki kod bloğu OpenRouter API formatına göre düzenlenmiştir.
*(API anahtarı için: https://openrouter.ai/keys)*

In [ ]:
# OpenRouter Alternatifi (Kullanmak için yorum satırlarını kaldırın)

# !pip install -q openai

# from openai import OpenAI
# OPENROUTER_KEY = "sk-or-..."

# or_client = OpenAI(
#     base_url="https://openrouter.ai/api/v1",
#     api_key=OPENROUTER_KEY,
# )

# def generate_with_openrouter(prompt):
#     completion = or_client.chat.completions.create(
#         model="meta-llama/llama-3.1-8b-instruct:free", # Tamamen ücretsiz model
#         messages=[{"role": "user", "content": prompt}],
#         max_tokens=4096,
#         temperature=0.7,
#     )
#     return completion.choices[0].message.content